# 260410-4: 정규화(Regularization)

이번 실습에서는 **Overfitting 문제를 직접 관찰**한 뒤, 이를 해결하는 다양한 **Regularization** 기법을 구현하고 비교합니다.

## 학습 목표
- Overfitting이 실제로 어떻게 발생하는지 직접 확인하기
- **L2 Regularization (Weight Decay)**을 직접 구현하고 가중치 변화 관찰
- **Dropout**을 적용하고 train/eval 모드 차이 이해하기
- **Early Stopping** 개념을 시각적으로 이해하기
- 2D toy data에서 **decision boundary** 시각화로 regularization 효과 체감

## 구성
1. Overfitting 먼저 체감하기
2. L2 Regularization 직접 구현 + 가중치 변화 관찰
3. Dropout 실습
4. Early Stopping 개념
5. Regularization 기법 종합 비교
6. 2D Toy Data에서 Decision Boundary 시각화
7. 정리 + 실용 팁

### Recap: 오전 강의 핵심

> **Optimization + Regularization** — Regularization 파트

| 개념 | 핵심 |
|------|------|
| **Overfitting** | Train은 잘 맞추지만 Test에서 성능 하락 — **variance** 문제 |
| **Underfitting** | Train에서도 성능이 낮음 — **bias** 문제 |
| **Bias-Variance Tradeoff** | 모델 복잡도 ↑ → bias ↓ but variance ↑ — 최적점을 찾아야 함 |
| **Augmented Error** | $E_{aug} = E_{train} + \lambda \cdot \Omega(w)$ — 학습 시 regularization term 추가 |
| **L2 Regularization** | $\Omega(w) = \sum w_i^2$ — 가중치를 작게 유지 (weight decay) |
| **Dropout** | 학습 중 랜덤하게 뉴런을 끔 → 특정 뉴런에 의존하지 않는 학습 |
| **Early Stopping** | Val loss가 올라가면 학습 중단 — 가장 간단하고 효과적 |

이번 실습에서는 **Overfitting을 직접 관찰**한 뒤, L2 Regularization과 Dropout을 구현하고 비교합니다.

In [ ]:
# Colab environment check
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess
    gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print('GPU available:')
        print(gpu_info.stdout.split('\n')[8])
    else:
        print('No GPU detected. Go to Runtime > Change runtime type > GPU')
    print('Colab detected. Ready to go!')
else:
    print('Running locally (not Colab)')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import copy
from sklearn.datasets import make_moons

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# 데이터 로딩 (CIFAR-10)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_set, val_set = torch.utils.data.random_split(trainset, [45000, 5000],
                                                     generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)

# 모델 정의
class OneHiddenMLP(nn.Module):
    def __init__(self, input_dim=3*32*32, hidden_dim=256, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x):
        return self.net(x)

def evaluate(model, data_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(model(images), 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

def train_model(model, optimizer, train_loader, val_loader, criterion, device, num_epochs=20):
    """학습 후 loss/accuracy 기록 반환 (범용 헬퍼)"""
    loss_hist, train_acc_hist, val_acc_hist = [], [], []
    for epoch in range(num_epochs):
        model.train()
        el, nb = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            el += loss.item(); nb += 1
        loss_hist.append(el / nb)
        train_acc_hist.append(evaluate(model, train_loader, device))
        val_acc_hist.append(evaluate(model, val_loader, device))
    return loss_hist, train_acc_hist, val_acc_hist

## 1. Overfitting 먼저 체감하기

Regularization을 배우기 전에, **왜 필요한지** 먼저 확인해봅시다.

아무런 regularization 없이 모델을 오래 학습시키면 어떤 일이 벌어질까요?
- **Train accuracy**는 계속 올라가지만
- **Validation accuracy**는 어느 시점부터 정체하거나 떨어집니다

이것이 바로 **Overfitting** — 학습 데이터의 noise까지 외워버리는 현상입니다.

In [ ]:
# Overfitting 관찰: regularization 없이 20 에폭 학습
torch.manual_seed(42)
model_overfit = OneHiddenMLP().to(device)
optimizer = optim.Adam(model_overfit.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

loss_hist, train_hist, val_hist = train_model(
    model_overfit, optimizer, train_loader, val_loader, criterion, device, num_epochs=20)

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, 21), loss_hist, 'b-o', markersize=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (계속 감소)'); ax1.grid(True)

ax2.plot(range(1, 21), train_hist, 'b-o', label='Train', markersize=4)
ax2.plot(range(1, 21), val_hist, 'r-o', label='Validation', markersize=4)
ax2.fill_between(range(1, 21), train_hist, val_hist, alpha=0.2, color='gray')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Train vs Val Accuracy (gap이 벌어짐 = Overfitting!)')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()

gap = train_hist[-1] - val_hist[-1]
print(f'Train Acc: {train_hist[-1]:.4f}, Val Acc: {val_hist[-1]:.4f}')
print(f'Overfitting Gap: {gap:.4f}')
print('\n-> Train fits well but Val drops. Reducing this gap is the goal of regularization!')

## 2. L2 Regularization 직접 구현하기

### L2 Regularization (Weight Decay)
모델의 가중치가 너무 커지지 않도록 loss에 **가중치의 제곱합**을 패널티로 추가합니다.

$$L_{\text{total}} = L_{\text{data}} + \lambda \sum_i w_i^2$$

- $\lambda$ (lambda): regularization 강도를 조절하는 하이퍼파라미터
- $\lambda$가 크면 가중치가 작아지도록 강하게 제약 → underfitting 가능
- $\lambda$가 작으면 제약이 약함 → overfitting 가능

### weight_decay와의 관계
PyTorch optimizer의 `weight_decay` 파라미터는 유사한 효과를 줍니다.

> **주의**: Adam optimizer에서 `weight_decay`는 엄밀히 L2 regularization과 다릅니다.
> Adam은 gradient를 adaptive하게 스케일링하기 때문에, **AdamW** (decoupled weight decay)를 쓰는 것이 더 정확합니다.
> 실무에서는 `optim.AdamW`를 사용하는 것이 권장됩니다.

In [ ]:
def l2_regularization_loss(model, lambda_):
    """
    L2 Regularization loss를 계산하세요.
    모델의 모든 파라미터에 대해 제곱합을 구한 뒤 lambda_를 곱합니다.

    Args:
        model: nn.Module 모델
        lambda_: regularization 강도 (float)
    Returns:
        reg_loss: 스칼라 값 (L2 penalty)

    Hint: model.parameters()로 모든 파라미터를 순회할 수 있습니다.
          각 파라미터 p에 대해 (p**2).sum() 또는 p.norm()**2 을 사용하세요.
    """
    ############################################################################
    # TODO 1: 여기에 구현하세요 (~3줄)                                            #
    ############################################################################
    reg_loss = 0.0
    for p in model.parameters():
        reg_loss += (p ** 2).sum()
    reg_loss = lambda_ * reg_loss
    ############################################################################
    #                             END OF YOUR CODE                             #
    ############################################################################
    return reg_loss

In [ ]:
def train_with_reg(model, optimizer, train_loader, val_loader, criterion, device,
                   lambda_=0.0, num_epochs=20):
    """L2 regularization을 적용하여 학습하고 기록을 반환"""
    loss_hist, train_acc_hist, val_acc_hist = [], [], []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss, num_batches = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            data_loss = criterion(outputs, labels)

            ####################################################################
            # TODO 2: data_loss에 L2 regularization을 추가하세요 (~2줄)          #
            # l2_regularization_loss 함수를 사용하세요                            #
            ####################################################################
            reg_loss = l2_regularization_loss(model, lambda_)
            total_loss = data_loss + reg_loss
            ####################################################################
            #                         END OF YOUR CODE                         #
            ####################################################################

            total_loss.backward()
            optimizer.step()
            epoch_loss += data_loss.item()
            num_batches += 1

        loss_hist.append(epoch_loss / num_batches)
        train_acc_hist.append(evaluate(model, train_loader, device))
        val_acc_hist.append(evaluate(model, val_loader, device))

    return loss_hist, train_acc_hist, val_acc_hist

### 가중치 분포 변화 관찰

L2 regularization은 "가중치를 작게 유지하라"는 제약입니다.
실제로 regularization 강도에 따라 학습 후 가중치가 얼마나 달라지는지 확인해봅니다.

In [ ]:
# 다양한 lambda로 학습 후 가중치 분포 비교
lambdas_demo = [0, 1e-3, 1e-2, 1e-1]
criterion = nn.CrossEntropyLoss()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, lam in zip(axes, lambdas_demo):
    torch.manual_seed(42)
    model_tmp = OneHiddenMLP().to(device)
    opt_tmp = optim.Adam(model_tmp.parameters(), lr=0.001)
    train_with_reg(model_tmp, opt_tmp, train_loader, val_loader, criterion, device,
                   lambda_=lam, num_epochs=10)

    # 첫 번째 Linear layer의 가중치 분포
    w = model_tmp.net[1].weight.data.cpu().flatten().numpy()
    ax.hist(w, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'λ = {lam}\nstd={w.std():.4f}', fontsize=11)
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlim(-0.15, 0.15)

plt.suptitle('Weight Distribution after Training by L2 Strength', fontsize=14)
plt.tight_layout()
plt.show()
print('-> Larger lambda pushes weights closer to 0 (weight decay!)')

## 3. Dropout 실습

**Dropout**은 딥러닝에서 가장 많이 사용되는 regularization 기법 중 하나입니다.

### 핵심 아이디어
학습 중 랜덤하게 일부 뉴런을 **꺼버림(drop)** → 특정 뉴런에 의존하지 않는 robust한 표현 학습

```
[학습 시 (model.train())]             [추론 시 (model.eval())]
○ ● ○ ● ○  ← 랜덤하게 끔              ● ● ● ● ●  ← 모두 사용
 \ |   | /   (p=0.5면 50% drop)         \ | | | /    (출력에 (1-p)를 곱함)
  ● ○ ●                                  ● ● ●
```

> **중요**: `model.train()`과 `model.eval()`에서 동작이 다릅니다!
> - `train()`: 랜덤하게 뉴런을 drop
> - `eval()`: 모든 뉴런 사용 (drop 없음, 대신 출력 스케일 조정)

In [ ]:
# Dropout이 있는 MLP
class MLPWithDropout(nn.Module):
    def __init__(self, input_dim=3*32*32, hidden_dim=256, num_classes=10, dropout_p=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),      # ← Dropout 추가!
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)

# train/eval 모드에서 Dropout 동작 차이 확인
model_drop = MLPWithDropout(dropout_p=0.5).to(device)
sample_input = torch.randn(1, 3, 32, 32).to(device)

model_drop.train()
out_train1 = model_drop(sample_input)
out_train2 = model_drop(sample_input)

model_drop.eval()
out_eval1 = model_drop(sample_input)
out_eval2 = model_drop(sample_input)

print("train() mode: different output each time (random drop)")
print(f"  output 1: {out_train1.data.flatten()[:5]}")
print(f"  output 2: {out_train2.data.flatten()[:5]}")
print(f"  diff:     {(out_train1 - out_train2).abs().max().item():.4f}")
print(f"\neval() mode: same output every time (no drop)")
print(f"  output 1: {out_eval1.data.flatten()[:5]}")
print(f"  output 2: {out_eval2.data.flatten()[:5]}")
print(f"  diff:     {(out_eval1 - out_eval2).abs().max().item():.6f}")

## 4. Early Stopping 개념

**Early Stopping**은 validation loss가 더 이상 줄지 않을 때 학습을 멈추는 전략입니다.

- 구현이 매우 간단 → 실무에서 가장 많이 사용되는 regularization
- 학습 시간도 절약됨
- `patience`: val loss가 개선되지 않는 에폭 수를 몇 번까지 참을지

아래에서 Section 1의 학습 기록을 다시 보며, 어느 시점에서 멈춰야 했는지 확인합니다.

In [ ]:
# Section 1에서 기록한 val_hist를 활용
# 최적의 stopping point 찾기
best_epoch = np.argmax(val_hist) + 1
best_val = max(val_hist)

fig, ax = plt.subplots(figsize=(10, 5))
epochs = range(1, len(val_hist) + 1)

ax.plot(epochs, train_hist, 'b-o', label='Train Acc', markersize=5)
ax.plot(epochs, val_hist, 'r-o', label='Val Acc', markersize=5)
ax.axvline(x=best_epoch, color='green', linestyle='--', linewidth=2,
           label=f'Best Val Acc (epoch {best_epoch})')
ax.fill_between(epochs, train_hist, val_hist, alpha=0.15, color='gray')

# patience 영역 표시
if best_epoch < len(val_hist):
    ax.axvspan(best_epoch, len(val_hist), alpha=0.1, color='red', label='Overfitting 영역')

ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Early Stopping: 언제 멈춰야 할까?')
ax.legend(loc='lower right'); ax.grid(True)
plt.tight_layout()
plt.show()

print(f'Best Epoch: {best_epoch} (Val Acc: {best_val:.4f})')
print(f'Final Epoch: {len(val_hist)} (Val Acc: {val_hist[-1]:.4f})')
print(f'-> Training beyond epoch {best_epoch} actually hurts performance!')

## 5. Regularization 기법 종합 비교

지금까지 배운 기법들을 모두 적용해서 비교해봅니다:
- **Baseline**: regularization 없음
- **L2 Only**: L2 regularization (λ=0.001)
- **Dropout Only**: Dropout (p=0.5)
- **L2 + Dropout**: 두 기법을 함께 사용

In [ ]:
criterion = nn.CrossEntropyLoss()
num_epochs = 20
comparison = {}

# 1) Baseline (no regularization)
torch.manual_seed(42)
m1 = OneHiddenMLP().to(device)
o1 = optim.Adam(m1.parameters(), lr=0.001)
l1, t1, v1 = train_model(m1, o1, train_loader, val_loader, criterion, device, num_epochs)
comparison['Baseline'] = {'train': t1, 'val': v1}

# 2) L2 Only
torch.manual_seed(42)
m2 = OneHiddenMLP().to(device)
o2 = optim.Adam(m2.parameters(), lr=0.001)
l2, t2, v2 = train_with_reg(m2, o2, train_loader, val_loader, criterion, device,
                              lambda_=0.001, num_epochs=num_epochs)
comparison['L2 (λ=0.001)'] = {'train': t2, 'val': v2}

# 3) Dropout Only
torch.manual_seed(42)
m3 = MLPWithDropout(dropout_p=0.5).to(device)
o3 = optim.Adam(m3.parameters(), lr=0.001)
l3, t3, v3 = train_model(m3, o3, train_loader, val_loader, criterion, device, num_epochs)
comparison['Dropout (p=0.5)'] = {'train': t3, 'val': v3}

# 4) L2 + Dropout (train_with_reg 사용을 위해 별도 루프)
torch.manual_seed(42)
m4 = MLPWithDropout(dropout_p=0.5).to(device)
o4 = optim.Adam(m4.parameters(), lr=0.001)
l4, t4, v4 = train_with_reg(m4, o4, train_loader, val_loader, criterion, device,
                              lambda_=0.001, num_epochs=num_epochs)
comparison['L2 + Dropout'] = {'train': t4, 'val': v4}

# 결과 출력
print(f"{'Method':<20s} | {'Train Acc':>10s} | {'Val Acc':>10s} | {'Gap':>8s}")
print("-" * 55)
for name, data in comparison.items():
    gap = data['train'][-1] - data['val'][-1]
    print(f"{name:<20s} | {data['train'][-1]:>10.4f} | {data['val'][-1]:>10.4f} | {gap:>8.4f}")

In [ ]:
# 종합 비교 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = {'Baseline': 'red', 'L2 (λ=0.001)': 'blue', 'Dropout (p=0.5)': 'green', 'L2 + Dropout': 'purple'}
epochs = range(1, num_epochs + 1)

for name, data in comparison.items():
    c = colors[name]
    ax1.plot(epochs, data['val'], '-o', color=c, label=name, markersize=4)
    ax2.plot(epochs, [t - v for t, v in zip(data['train'], data['val'])],
             '-o', color=c, label=name, markersize=4)

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Accuracy')
ax1.set_title('Validation Accuracy'); ax1.legend(); ax1.grid(True)

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Train - Val Gap')
ax2.set_title('Overfitting Gap 비교 (작을수록 좋음)'); ax2.legend(); ax2.grid(True)
ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 2D Toy Data에서 Decision Boundary 시각화

CIFAR-10은 고차원(3072차원)이라 decision boundary를 직접 볼 수 없습니다.
2D toy data를 사용하면 regularization이 decision boundary에 미치는 영향을 **직접 눈으로** 확인할 수 있습니다.

- `make_moons`: 반달 모양의 2D 이진 분류 데이터
- 큰 MLP (2→64→64→2)를 사용하여 의도적으로 overfitting할 수 있는 환경을 만듭니다.

In [ ]:
# 2D Moon 데이터 생성
np.random.seed(42)
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)

plt.figure(figsize=(6, 4))
plt.scatter(X_moon[y_moon==0, 0], X_moon[y_moon==0, 1], c='blue', label='Class 0', edgecolors='k')
plt.scatter(X_moon[y_moon==1, 0], X_moon[y_moon==1, 1], c='red', label='Class 1', edgecolors='k')
plt.title('Moon Dataset'); plt.legend(); plt.grid(True)
plt.show()

X_tensor = torch.FloatTensor(X_moon).to(device)
y_tensor = torch.LongTensor(y_moon).to(device)

In [ ]:
# 2D MLP + Decision Boundary 시각화 함수
class MLP2D(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        layers = [nn.Linear(2, 64), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(p=dropout_p))
        layers += [nn.Linear(64, 64), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(p=dropout_p))
        layers.append(nn.Linear(64, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def plot_decision_boundary(model, X, y, ax, title=''):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).to(device)

    model.eval()
    with torch.no_grad():
        Z = torch.softmax(model(grid), dim=1)[:, 1].cpu().numpy()
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.8)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', edgecolors='k', s=30)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red', edgecolors='k', s=30)
    ax.set_title(title, fontsize=12)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

In [ ]:
# 4가지 조건으로 2D 모델 학습 및 decision boundary 비교
criterion = nn.CrossEntropyLoss()
configs = {
    'No Reg': {'lambda_': 0, 'dropout': 0.0},
    'L2 (λ=0.001)': {'lambda_': 0.001, 'dropout': 0.0},
    'Dropout (p=0.5)': {'lambda_': 0, 'dropout': 0.5},
    'L2 + Dropout': {'lambda_': 0.001, 'dropout': 0.5},
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, cfg) in zip(axes, configs.items()):
    torch.manual_seed(42)
    model_2d = MLP2D(dropout_p=cfg['dropout']).to(device)
    optimizer = optim.Adam(model_2d.parameters(), lr=0.01)

    for epoch in range(300):
        model_2d.train()
        optimizer.zero_grad()
        outputs = model_2d(X_tensor)
        loss = criterion(outputs, y_tensor)
        if cfg['lambda_'] > 0:
            loss = loss + l2_regularization_loss(model_2d, cfg['lambda_'])
        loss.backward()
        optimizer.step()

    model_2d.eval()
    with torch.no_grad():
        _, pred = torch.max(model_2d(X_tensor), 1)
        acc = (pred == y_tensor).float().mean().item()

    plot_decision_boundary(model_2d, X_moon, y_moon, ax, title=f'{name}\nAcc: {acc:.2f}')

plt.suptitle('Decision Boundary by Regularization', fontsize=14)
plt.tight_layout()
plt.show()

## 7. 정리 + 실용 팁

### Bias-Variance Tradeoff
위 시각화에서 관찰할 수 있는 것:
- **No Reg**: boundary가 매우 복잡 → noise까지 학습 (overfitting, high variance)
- **적절한 Reg**: boundary가 부드러워지면서도 데이터를 잘 분류
- **과도한 Reg**: boundary가 너무 단순 → 데이터 구조를 못 잡음 (underfitting, high bias)

**핵심**: 적절한 regularization 강도를 찾는 것이 중요하며, 이를 위해 **validation set**을 활용합니다.

### Regularization 기법 요약

| 기법 | 핵심 아이디어 | PyTorch 구현 |
|------|------------|-------------|
| **L2 (Weight Decay)** | 가중치 크기 제한 | `optimizer(weight_decay=...)` 또는 직접 loss에 추가 |
| **Dropout** | 랜덤하게 뉴런 비활성화 | `nn.Dropout(p=0.5)` |
| **Early Stopping** | val loss 기준으로 학습 중단 | 직접 구현 또는 callback 사용 |
| **Data Augmentation** | 학습 데이터 변형으로 다양성 증가 | `transforms`, Albumentations |
| **Batch Normalization** | 층별 입력 정규화 (regularization 효과도 있음) | `nn.BatchNorm1d/2d` |

### 실용 팁
- **AdamW** (`optim.AdamW`): Adam + 올바른 weight decay 구현. 실무 표준.
- **PyTorch Lightning**: `EarlyStopping` callback으로 early stopping을 깔끔하게 구현 가능
- **Weight Decay 기본값**: 보통 `1e-2` ~ `1e-4` 범위에서 시작
- **Dropout 위치**: 보통 큰 hidden layer 뒤에 적용, 마지막 output layer 앞에는 넣지 않음